# ModelListGP

`ModelListGP` combines independent GP models into one multi-output model.

This is useful when outcomes:

- have different training inputs,
- should not share a cross-output covariance model,
- or are naturally modeled by separate GP instances.

It is also a common building block for multi-objective Bayesian optimization.


## 1. Imports and reproducibility


In [ ]:
import matplotlib.pyplot as plt
import torch
from botorch.fit import fit_gpytorch_mll

from robotorchan.models import ModelListGP, SingleTaskGP

torch.manual_seed(0)
dtype = torch.double


## 2. Synthetic outputs with different training inputs


In [ ]:
def objective_1(x: torch.Tensor) -> torch.Tensor:
    return torch.sin(2 * torch.pi * x)

def objective_2(x: torch.Tensor) -> torch.Tensor:
    return 0.6 * torch.cos(2 * torch.pi * x) + 0.2 * x

train_X1 = torch.linspace(0.05, 0.95, 10, dtype=dtype).unsqueeze(-1)
train_X2 = torch.linspace(0.10, 0.90, 7, dtype=dtype).unsqueeze(-1)

train_Y1 = objective_1(train_X1) + 0.03 * torch.randn_like(train_X1)
train_Y2 = objective_2(train_X2) + 0.03 * torch.randn_like(train_X2)


## 3. Build child models and `ModelListGP`


In [ ]:
model_1 = SingleTaskGP(train_X1, train_Y1)
model_2 = SingleTaskGP(train_X2, train_Y2)
model = ModelListGP(model_1, model_2)

print("number of child models:", len(model.models))
print("supports_mll:", model.supports_mll)
print("raw_train_Xs shapes:", [x.shape for x in model.raw_train_Xs])
print("raw_train_Ys shapes:", [y.shape for y in model.raw_train_Ys])


`ModelListGP` intentionally stores raw data per child model. It does **not** invent a single container-level `raw_train_X`.


## 4. Fit all child models together


In [ ]:
mll = model.make_mll()
print(type(mll).__name__)

fit_gpytorch_mll(mll)


## 5. Multi-output posterior


In [ ]:
test_X = torch.linspace(0.0, 1.0, 200, dtype=dtype).unsqueeze(-1)

model.eval()
with torch.no_grad():
    posterior = model.posterior(test_X)
    mean = posterior.mean
    variance = posterior.variance

print("posterior mean shape:", mean.shape)


In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))

ax.scatter(train_X1.squeeze(-1), train_Y1.squeeze(-1), label="output 1 observations")
ax.scatter(train_X2.squeeze(-1), train_Y2.squeeze(-1), label="output 2 observations")
ax.plot(test_X.squeeze(-1), mean[..., 0], label="output 1 posterior")
ax.plot(test_X.squeeze(-1), mean[..., 1], label="output 2 posterior")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_title("ModelListGP with independent outputs")
ax.legend()
plt.show()


## 6. Why `ModelListGP` instead of a multi-task GP?

`ModelListGP` assumes the child models are independent. It is appropriate when:

- outputs have different observation locations,
- cross-output correlation does not need to be modeled,
- different GP types or transforms should be used per output,
- a multi-objective BO problem needs separate surrogate models.

Use `MultiTaskGP` or `KroneckerMultiTaskGP` when learning task-to-task covariance is part of the modeling goal.


## 7. BO connection

A `ModelListGP` can be passed to multi-output Monte Carlo acquisition functions. In multi-objective BO, the next layer is typically an acquisition such as qEHVI / qNEHVI together with an objective or reference point.

This notebook focuses on the model layer; dedicated multi-objective acquisition examples can be added separately.
